In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from delta.tables import DeltaTable
import uuid

spark = SparkSession.builder.getOrCreate()
meu_batch_id = str(uuid.uuid4())
volume_path = "/Volumes/workspace/default/raw_data"

def processa_fatos_bronze_prata(nome_tabela, chave_primaria, arquivo_csv):
    print(f"--- Processando {nome_tabela} ---")
    
    # ==========================================
    # 1. CAMADA BRONZE (Ingestão com Governança)
    # ==========================================
    df_raw = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/{arquivo_csv}")
    
    colunas_originais = df_raw.columns
    df_bronze = (
        df_raw
        .withColumn("arquivo_origem", F.col("_metadata.file_path"))
        .withColumn("data_ingestao", F.current_date())
        .withColumn("timestamp_ingestao", F.current_timestamp())
        .withColumn("batch_id", F.lit(meu_batch_id))
        .withColumn("hash_linha", F.md5(F.concat_ws("||", *[F.col(c).cast("string") for c in colunas_originais])))
        .withColumn("schema_version", F.lit("1.0"))
    )
    
    tabela_bronze = f"workspace.default.bronze_{nome_tabela}"
    df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(tabela_bronze)
    print(f"Bronze OK: {tabela_bronze}")

    # ==========================================
    # 2. CAMADA PRATA (Deduplicação e Idempotência)
    # ==========================================
    # Filtra o lote atual
    df_lote = spark.read.table(tabela_bronze).filter(F.col("batch_id") == meu_batch_id)
    
    # Deduplicação usando ROW_NUMBER (Requisito Sênior)
    # Se vierem 2 transações com mesmo ID, pega a que foi ingerida por último
    window_spec = Window.partitionBy(chave_primaria).orderBy(F.col("timestamp_ingestao").desc())
    df_upsert = df_lote.withColumn("row_num", F.row_number().over(window_spec)) \
                       .filter(F.col("row_num") == 1).drop("row_num")

    tabela_prata = f"workspace.default.silver_{nome_tabela}"
    
    if not spark.catalog.tableExists(tabela_prata):
        df_upsert.write.format("delta").saveAsTable(tabela_prata)
        print(f"Prata OK: {tabela_prata} criada.")
    else:
        # MERGE Idempotente (Insere apenas transações/eventos novos)
        delta_table = DeltaTable.forName(spark, tabela_prata)
        insert_cond = f"target.{chave_primaria} = source.{chave_primaria}"
        
        delta_table.alias("target").merge(
            df_upsert.alias("source"), insert_cond
        ).whenNotMatchedInsertAll().execute()
        print(f"Prata OK: {tabela_prata} atualizada via MERGE.")

# Execução do Pipeline para os 3 domínios
processa_fatos_bronze_prata("transacoes", "id_transacao", "transacoes.csv")
processa_fatos_bronze_prata("eventos_risco", "id_evento", "eventos_risco.csv")
processa_fatos_bronze_prata("estornos", "id_estorno", "estornos.csv")